# C1–C6  BKT computational verification for QVC / TEVC vortex RG

The locked amplitude theory is a saddle-node fold at finite $\Delta_c$. A 2D phase-vortex sector is a physically interesting addition: if the superfluid stiffness $\kappa(\alpha)$ is driven through the Kosterlitz surface $K=2/\pi$ *before* the fold, hypothesis H-VRG1 would convert the infrared of the *phase* into a BKT unbinding problem, with a target gap $\Delta\sim\hbar c_s/\xi_{\mathrm{KT}}$.

This notebook runs the numerical-analytic suite **C1→C6 in order**, using Option A* locks and existing GPE / fold solvers. It does **not** wait for a preprint rewrite.

**Defect vocabulary.** A hopfion (toroidal Casimir cavity, $Q_H=1/N$) is not a KT vortex. A 2D phase vortex is a point defect with $\oint\nabla\theta\cdot d\mathbf{l}=2\pi n$. A Chern–Simons fluxon binds flux $2\pi/N$. The three objects are never identified.

**Closures.** Option A* freezes the cavity: $\alpha$ does not run with the KT scale $\ell$. Chern–Simons is truncated unless an anyon $K_c$ scan is switched on as a labeled option. Democratic $\lambda_{\min}=-g_0$. C1 is partly ansatz (density unit $n_0$, Peotta–Törmä geometric floor), so this notebook does **not** claim a BKT theorem.

The old diagnostic $\kappa\propto\Delta^2$ together with an input $T_{\mathrm{BKT}}^{\mathrm{bare}}=1\,\mathrm{K}$ is not used: it assumes the answer.

In [ ]:
import sys
from pathlib import Path
from IPython.display import display, Image, Markdown

ROOT = Path.cwd().resolve()
if ROOT.name == "bkt_suite":
    ROOT = ROOT.parents[1]
elif (ROOT / "notebooks" / "bkt_suite").is_dir():
    pass
else:
    ROOT = Path("/Users/mineralhoiland/Code/QVCCursor")
sys.path.insert(0, str(ROOT))

from qvc_bkt.bootstrap import ensure_qvccompute
ensure_qvccompute()

from qvc_bkt.locks import build_bkt_locks
from qvc_bkt.stiffness import run_c1
from qvc_bkt.vortex_core import run_c2
from qvc_bkt.kt_flow import run_c3
from qvc_bkt.xi_kt import run_c4
from qvc_bkt.model_compare import run_c5
from qvc_bkt.phonon_damping import run_c6
from qvc_bkt.suite import (
    DEFAULT_OUT,
    computed_vs_assumed,
    h_vrg1_statement,
    key_numbers,
    render_markdown,
)
from qvc_bkt.figures import write_all_figures
from qvc_bkt.kt_flow import competing_scale_scan
from qvc_bkt.xi_kt import build_vortex_gap_curve
from qvc_bkt.model_compare import compare_models
import numpy as np

OUT = ROOT / "notebooks" / "bkt_suite" / "output"
OUT.mkdir(parents=True, exist_ok=True)
locks = build_bkt_locks()
print("Option A* fat torus")
print(f"  (r,R)=({locks.r_um*1e3:.0f},{locks.R_um*1e3:.0f}) nm   ħc_s={locks.hbar_cs_meV_um} meV·µm")
print(f"  F={locks.F:.5f}   E_vac={locks.E_vac_meV:.4f} meV")
print(f"  α_c=δ_c={locks.alpha_c:.4f}   Δ_c={locks.Delta_c_meV:.4f} meV   Δ_0={locks.Delta0_meV:.3f} meV")
print(f"  λ★={locks.lambda_star:.4f}   Δ★={locks.Delta_star_meV:.4f} meV   T*={1e3*locks.T_star_K:.1f} mK")
print(f"  E_pair={locks.E_pair_meV:.4f} meV   λ_min=-g_0={locks.lambda_min_meV:.4f} meV   Q_H={locks.Q_H}")
print(f"  CS status: {locks.cs_status}   K_c(bosonic)={locks.k_c_bosonic:.4f}   K_c(anyon)={locks.k_c_anyon:.4f}")

## C1  Microscopic-leaning stiffness $\kappa(\alpha)$

Two-sector identity (**derived** given the polar split of the bulk TQGL energy):

$$\kappa\equiv\rho_s=2\kappa_{\mathrm{GL}}\langle\rho^2\rangle+\kappa_{\mathrm{geom}}.$$

- $\kappa_{\mathrm{GL}}=\Delta_0 a_H^2$ from the truncated GPE kinetic coefficient, $a_H=r$ (Option A*).
- $\langle\rho^2\rangle=[\delta_+(\alpha)]^2$ from the locked physical branch (live-gap map $\Delta/\Delta_0=\sqrt{n/n_{\mathrm{ref}}}$).
- $n_0=1/(\pi a_H^2)$ converts dimensionless GPE density into a 2D density so $\kappa$ has energy units (**ansatz**). Then $\kappa_{\mathrm{conv}}(0)=2\Delta_0/\pi$.
- $\kappa_{\mathrm{geom}}=(C_{\mathrm{eff}}/2\pi)\Delta_0$ is a Peotta–Törmä-type **frozen** geometric floor (**ansatz**). Primary $C_{\mathrm{eff}}=1$; $C_{\mathrm{eff}}=Q_H$ is a labeled variant, not an identification of hopfion with vortex.
- A uniform phase twist checks the diamagnetic current–current piece (**computed**).

Kosterlitz convention: $K=\kappa/T_{\mathrm{eff}}$, critical point $K_c=2/\pi$. $T_{\mathrm{eff}}=T^*$ and a small $T$ grid. No $T_{\mathrm{BKT}}^{\mathrm{bare}}$ is inserted.

In [ ]:
c1 = run_c1(locks, n_alpha=80)
p = c1["primary"]
print("Primary closure:", c1["primary_closure"])
print(f"κ_GL = {p['kappa_GL_meV_um2']:.4e} meV·µm²")
print(f"n0 (healing disk) = {p['n0_um2']:.4g} µm⁻²")
print(f"κ(UV) = {p['kappa_uv_meV']:.4f} meV    κ(fold) = {p['kappa_fold_meV']:.4f} meV")
print(f"K(UV,T*) = {p['K_uv_Tstar']:.3f}    K(fold,T*) = {p['K_fold_Tstar']:.3f}    K_c = {p['K_c_bosonic']:.4f}")
print(f"twist rel. err. = {p['twist_uniform']['rel_err']:.3e}   ({p['twist_uniform']['status']})")
print(f"hopfion ring ⟨ρ²⟩ = {p['hopfion_rho2']['rho2_hopfion_ring']:.4f}  [{p['hopfion_rho2']['defect']}]")
print("Variants (κ_UV, κ_fold, K_fold at T*):")
for name, v in c1["variants"].items():
    print(f"  {name:18s}  κ_fold={v['kappa_fold_meV']:.4f} meV   K_fold={v['K_fold_Tstar']:.3f}")

## C2  Core energy and $a_c$

Relax a **winding-1 2D GP vortex** in the truncated cubic-quintic GPE (Chern–Simons, Berry, Skyrme off). This is not a hopfion.

$$E_v=\pi\kappa\ln(R/a_c)+E_{\mathrm{core}}.$$

The IR logarithm uses $a_c=\xi$ (healing length, **ansatz cutoff**). The half-density radius is **computed**. $E_{\mathrm{core}}/\kappa$ is computed given that cutoff. The geometric floor is a band property and is not inserted into the GP functional; the log coefficient is the conventional two-sector $\kappa$.

In [ ]:
kappa_gp_uv = c1["variants"]["conv_only"]["kappa_uv_meV"]
c2 = run_c2(locks, kappa_uv_meV=kappa_gp_uv)
print("defect:", c2["defect"], " hopfion?", not c2["radial"]["not_hopfion"])
print(f"BVP success: {c2['radial']['bvp_success']}  ({c2['radial']['bvp_message']})")
print(f"ξ = {c2['radial']['xi_um']*1e3:.3f} nm    R = {c2['radial']['R_um']*1e3:.1f} nm")
print(f"a_c (used, {c2['a_c_label']}) = {c2['a_c_um']*1e3:.3f} nm")
print(f"a_{{1/2}} (computed) = {c2['core']['a_c_half_um']*1e3:.3f} nm")
print(f"E_core/κ = {c2['E_core_over_kappa']:.4f}")
print(f"E_core/κ using a_{{1/2}} cutoff = {c2['core']['E_core_over_kappa_using_a_half']:.4f}")

## C3  Kosterlitz integration at frozen $\alpha$

$$\frac{\mathrm{d}K^{-1}}{\mathrm{d}\ell}=4\pi^3 y^2,\qquad
\frac{\mathrm{d}y}{\mathrm{d}\ell}=(2-\pi K)y,\qquad
y=\exp(-E_{\mathrm{core}}/T_{\mathrm{eff}}).$$

No $\beta_\alpha$: Option A* freezes the cavity. H-VRG1 holds iff the physical drive hits $K=2/\pi$ (or IR plasma) **before** $\alpha=\alpha_c$.

The anyon line $K_c=(2/\pi)(1-1/N)^2$ is a **labeled CS-restored variant**, not mixed into the truncated default.

In [ ]:
c3 = run_c3(c1, c2, locks)
h = c3["h_vrg1_Tstar_primary"]
print("E_core/κ used:", c3["E_core_over_kappa_used"], f"({c3['E_core_over_kappa_label']})")
print("H-VRG1 at T*, primary closure: KT before fold =", h["kt_before_fold"], " fold wins =", h["fold_wins"])
print("  conv-only at T*:     ", c3["scan_Tstar_conv_only"]["kt_before_fold"],
      "  α_KT/α_c =", c3["scan_Tstar_conv_only"]["alpha_kt_bare_over_ac"])
print("  anyon K_c at T*:     ", c3["scan_Tstar_anyon"]["kt_before_fold"],
      "  (CS restored, labeled)")
print("T_eff grid (primary | conv-only):")
for row in c3["T_eff_grid"]:
    print(f"  T={row['T_K']:.4g} K   primary KT-first={row['primary_kt_before_fold']}   "
          f"conv KT-first={row['conv_only_kt_before_fold']}")
print("Station flows at T*:")
for st in c3["flows_Tstar"]:
    print(f"  {st['name']:8s}  K_bare={st['K_bare']:.3f}  y_bare={st['y_bare']:.2e}  plasma={st['flow']['plasma']}")

## C4  Continuous $\Delta\to 0$ diagnostic

If unbinding occurs, $\xi_{\mathrm{KT}}=a_c e^{\ell_\ast}$ from the flow, and the H-VRG1 *target*

$$\Delta\sim\hbar c_s/\xi_{\mathrm{KT}}$$

is compared to locked finite $\Delta_c$. This is a numerical implication of the vortex sector (**conjecture** as a gap identification). It does not replace the fold theorem.

If the primary $T^*$ scan stays bound, a labeled counterfactual at $T=1\,\mathrm{K}$ (conventional $\kappa$ only) is constructed so the $\xi_{\mathrm{KT}}$ map can still be inspected.

In [ ]:
c4 = run_c4(c3, c2, locks)
print("Unbinding at T* (primary)?", c4["primary_Tstar"]["unbinding_occurs"])
print("Unbinding at T* (conv-only)?", c4["conv_only_Tstar"]["unbinding_occurs"])
print("Locked Δ_c =", c4["vs_locked_Delta_c"]["Delta_c_locked_meV"], "meV")
print("Vortex Δ_min (if any) =", c4["vs_locked_Delta_c"]["Delta_vortex_min_meV"])
print(c4["vs_locked_Delta_c"]["note"])

if not c4["primary_Tstar"]["unbinding_occurs"]:
    sc1 = competing_scale_scan(
        c1["variants"]["conv_only"]["rows"],
        locks=locks,
        T_K=1.0,
        E_core_over_kappa=float(c3["E_core_over_kappa_used"]),
        ell_ir=float(c3["scan_Tstar_primary"]["ell_ir"]),
    )
    curve = build_vortex_gap_curve(sc1, locks=locks, a_c_um=float(c2["a_c_um"]), ell_ir=sc1["ell_ir"])
    c4["counterfactual_T1K_conv_only"] = {
        "scan": {k: sc1[k] for k in ("T_K", "kt_before_fold", "fold_wins", "alpha_kt_bare_over_ac", "K_c", "cs")},
        "curve": curve,
        "label": "counterfactual: T=1 K, conventional κ only",
    }
    print("Counterfactual T=1 K conv-only: KT before fold =", sc1["kt_before_fold"],
          " α_KT/α_c =", sc1["alpha_kt_bare_over_ac"])
    if curve["unbinding_occurs"]:
        rows = curve["rows"]
        tA = np.array([r["t_A"] for r in rows])
        dv = np.array([r["Delta_vortex_meV"] for r in rows]) / locks.Delta0_meV
        finite = np.array([r["finite_unbinding"] for r in rows], dtype=bool) & np.isfinite(dv)
        if np.count_nonzero(finite) >= 8:
            c4["counterfactual_T1K_conv_only"]["c5_preview"] = compare_models(
                tA[finite], dv[finite], 0.0, label="vortex_Delta_T1K_conv_only"
            )

## C5  Model comparison $\sqrt{t_A}$ vs $\exp(-C/\sqrt{t_A})$

On (i) the locked fold $\Delta(\alpha)$ and (ii) vortex-implied $\Delta$ from C4, over $\gtrsim 1.5$ decades in $t_A=(\alpha_c-\alpha)/\alpha_c$. Gaussian residual log-likelihood and free slope of $\ln(\delta-\delta_c)$ vs $\ln t_A$.

Fold-only C5 preferring square-root scaling is the locked amplitude theorem. It is **not** a BKT disproof.

In [ ]:
c5 = run_c5(c4, locks)
fc = c5["fold_comparison"]
print("Locked fold branch:")
print(f"  decades in t_A = {fc['sqrt']['tA_decades']:.2f}")
print(f"  free slope ln(δ−δ_c) vs ln t_A = {fc['sqrt']['slope_free']:.4f}  (theorem: 1/2)")
print(f"  Δℓℓ (√ minus essential) = {fc['delta_ll_sqrt_minus_essential']:.3f}  preferred: {fc['preferred']}")
print("Vortex-implied:", c5["vortex_note"])
if c5["vortex_comparison"]:
    vc = c5["vortex_comparison"]
    print(f"  preferred: {vc['preferred']}  Δℓℓ = {vc['delta_ll_sqrt_minus_essential']:.3f}")
cf = (c4.get("counterfactual_T1K_conv_only") or {}).get("c5_preview")
if cf:
    print("Counterfactual T=1 K conv-only vortex Δ:")
    print(f"  preferred: {cf['preferred']}  Δℓℓ = {cf['delta_ll_sqrt_minus_essential']:.3f}")
print(c5["disclaimer"])

## C6  Phonon damping / H-VRG2

AHNS overdamped criterion $\hbar/\tau_v\gtrsim\pi\kappa$. Two labeled estimates, not a microscopic mobility:

1. Existing GPE phonon potential: $\hbar/\tau_v\sim g_{\mathrm{ph}}=0.1\Delta_0$.
2. Gao–Khalaf-type Eliashberg OOM: $\hbar/\tau_v\sim 2\pi\lambda_{\mathrm{ep}}k_BT$ with $\lambda_{\mathrm{ep}}=1$.

The retardation identity $\eta_{\mathrm{ret}}=2\Delta_0\xi/(\hbar c_s)$ is locked algebra and is not by itself a vortex damping rate. Report jump vs rounded crossover along the physical branch.

In [ ]:
c6 = run_c6(c1, c3, locks)
print("GPE phonon:", c6["gpe_phonon"]["formula"])
print(f"  ħ/τ_v = {c6['gpe_phonon']['hbar_over_tau_meV']:.4f} meV")
print("Eliashberg at T*:", c6["eliashberg_Tstar"]["formula"])
print(f"  ħ/τ_v = {c6['eliashberg_Tstar']['hbar_over_tau_meV']:.4f} meV")
print(f"η_ret = {c6['retardation']['eta_ret']:.4f}  ({c6['retardation']['status']})")
print("Anchors:")
for name, ah in c6["anchors"].items():
    print(f"  {name:24s}  ratio={ah['ratio']:.3g}  {ah['verdict']}")
print("Jump vs crossover:", c6["jump_vs_crossover"])
print(c6["h_vrg2_reading"])

## Figures and summary artifacts

Write matplotlib figures plus `summary.json` / `summary.md` of solved numerical parameters. Re-run this cell after C1–C6.

In [ ]:
import json
from qvc_bkt.suite import _jsonify, _drop_profiles

bundle = {
    "suite": "QVC / TEVC vortex RG  C1–C6",
    "locks": locks.as_dict(),
    "C1": c1, "C2": c2, "C3": c3, "C4": c4, "C5": c5, "C6": c6,
    "computed_vs_assumed": computed_vs_assumed(c1, c2, c3, c4, c5, c6, locks),
    "h_vrg1": h_vrg1_statement(c3, c1),
    "key_numbers": key_numbers(locks, c1, c2, c3, c4, c5, c6),
}
figs = write_all_figures(bundle, OUT)
bundle["figures"] = figs
(OUT / "summary.json").write_text(json.dumps(_jsonify(_drop_profiles(bundle)), indent=2))
(OUT / "summary.md").write_text(render_markdown(bundle))
print("Wrote", OUT)
for f in figs:
    print(" ", Path(f).name)

display(Markdown(render_markdown(bundle)))
for f in figs:
    display(Image(f))